In [5]:
"""
Fine-tune Whisper Model on Audio and Text Files in a Folder (Colab)

This code assumes you have a folder containing .wav audio files and corresponding
.txt files with the same base name containing the transcriptions.

**Before running:**

1.  **Upload your audio (.wav) and text (.txt) files to a folder in your Google Drive.**
    For example, create a folder named `medical_data` and place your files there.
    Ensure that for each `audio.wav` file, there is a corresponding `audio.txt`
    file in the same folder.
2.  **Grant Colab access to your Google Drive.**

**Important Notes:**

* This is a basic example and might require adjustments based on your specific
    dataset size, audio quality, and desired level of fine-tuning.
* Fine-tuning can be computationally intensive and may take a significant amount
    of time depending on the size of your data and the chosen model.
* Consider using a GPU runtime for faster training (Runtime -> Change runtime type -> GPU).
* This code uses a smaller Whisper model (`tiny`) for demonstration purposes. You
    can experiment with larger models (`small`, `base`, `medium`, `large-v2`, etc.)
    for potentially better accuracy, but they will require more memory and training time.
* The learning rate, number of epochs, and other hyperparameters might need
    tuning for optimal performance on your specific dataset.

"""

# Install necessary libraries
# !pip install -q datasets transformers accelerate jiwer

# Import libraries
import os, re
from datasets import Dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Trainer
from typing import Dict, List, Union
import soundfile as sf

In [6]:
output_model = "./self_trained_models/whisper-finetuned"
# --- 2. Load Audio Files and Transcriptions ---
data_folder = 'data_folder'  # Replace with the actual path to your folder
audio_files = []
transcriptions = []

def load_data(data_dir):
    audio_files = []
    texts = []
    pattern = re.compile(r'medical_terminology_\d{1}_\d+\.wav$')
    for file in os.listdir(data_dir):
        if pattern.match(file):
            txt_path = os.path.join(data_dir, file.replace(".wav", ".txt"))
            if os.path.exists(txt_path):
                with open(txt_path, "r", encoding="utf-8") as f:
                    text = f.read().strip()
                audio_files.append(os.path.join(data_dir, file))
                texts.append(text)
    return Dataset.from_dict({"audio": audio_files, "text": texts}).cast_column("audio", Audio(sampling_rate=16000))


dataset = load_data("data_folder").train_test_split(test_size=0.2)

In [7]:
# --- 3. Load Whisper Processor ---
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")

In [8]:
# ---4. prepare data sets ---
def prepare_dataset(batch, processor):
    """Prepare dataset by converting audio to features and text to tokens."""
    # Process audio
    audio = batch["audio"]
    inputs = processor(
        audio["array"], 
        sampling_rate=audio["sampling_rate"], 
        return_tensors="pt"
    )
    
    # Process text
    labels = processor.tokenizer(
        batch["text"],
        padding=False,
        truncation=True,
        max_length=448,  # Whisper max context size
        return_tensors="pt"
    )
    
    # Replace padding token id's of the labels by -100 so it's ignored by the loss function
    labels = labels["input_ids"][0]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    
    # Store as tensors (not nested lists)
    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = labels
    
    return batch


train_dataset = dataset["train"].map(
    lambda batch: prepare_dataset(batch, processor),
    remove_columns=dataset["train"].column_names
)

eval_dataset = dataset["test"].map(
    lambda batch: prepare_dataset(batch, processor),
    remove_columns=dataset["test"].column_names
)

Map: 100%|██████████| 50/50 [00:02<00:00, 22.46 examples/s]


In [ ]:
# ---5. Collator ---
import torch
from dataclasses import dataclass

@dataclass
class WhisperDataCollator:
    processor: WhisperProcessor
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Extract input features and labels
        input_features = []
        labels = []
        for feature in features:
            input_feat = feature["input_features"]
            label = feature["labels"]
            # Ensure input_features is a tensor
            if isinstance(input_feat, list):
                input_feat = torch.tensor(input_feat)
            # Ensure labels is a tensor
            if isinstance(label, list):
                label = torch.tensor(label)
            
            input_features.append(input_feat)
            labels.append(label)
        # Stack input features
        batch_input_features = torch.stack(input_features)
        # Pad labels to the same length
        batch_labels = torch.nn.utils.rnn.pad_sequence(
            labels, 
            batch_first=True, 
            padding_value=-100
        )
        return {
            "input_features": batch_input_features,
            "labels": batch_labels }

In [10]:
# ---6.Training ---
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-finetuned",
    per_device_train_batch_size=8,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=50,
    learning_rate=1e-5,
    generation_max_length=225,
    predict_with_generate=True,
    report_to="none",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=WhisperDataCollator(processor),
    tokenizer=processor.tokenizer
)

trainer.train()
trainer.save_model(output_model)
processor.save_pretrained(output_model)

C:\Users\HP\AppData\Local\Temp\ipykernel_13584\627317958.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\HP\Saved Games\android\python\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss


c:\Users\HP\Saved Games\android\python\Lib\site-packages\transformers\modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


[]

In [2]:
class MemoryEfficientEvaluator:
    
    def __init__(self, model_path: str = "./whisper-finetuned"):
        """Initialize the evaluator with memory-efficient settings."""
        print("Loading model for evaluation...")
        
        # Force garbage collection before loading model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Load with memory-efficient settings
        self.processor = WhisperProcessor.from_pretrained(model_path)
        self.model = WhisperForConditionalGeneration.from_pretrained(
            model_path,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            low_cpu_mem_usage=True
        )
        
        # Move to GPU if available
        if torch.cuda.is_available():
            self.model = self.model.cuda()
            print("Using GPU for inference")
        else:
            print("Using CPU for inference")
        
        self.model.eval()
        
        # Filler words for cleaning
        self.filler_words = {
            "um", "uh", "like", "so", "well", "you know", 
            "ah", "er", "eh", "hmm", "mm", "uhh", 
        }
    
    def clean_text(self, text: str) -> str:
        """Clean text by removing filler words and normalizing."""
        # Convert to lowercase
        text = text.lower()
        
        # Remove filler words
        for word in self.filler_words:
            text = re.sub(rf'\b{word}\b', '', text, flags=re.IGNORECASE)
        
        # Remove extra whitespace and punctuation
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        
        return text
    
    def load_audio_efficiently(self, audio_path: str):
        """Load audio with memory-efficient chunked reading."""
        try:
            # First, get audio info without loading the entire file
            info = sf.info(audio_path)
            print(f"Audio info: {info.frames} frames, {info.samplerate} Hz, {info.duration:.2f}s")
            
            # Calculate max frames for 30 seconds
            max_frames = int(30 * info.samplerate)
            frames_to_read = min(info.frames, max_frames)
            
            # Read only the frames we need
            audio, sr = sf.read(audio_path, frames=frames_to_read, dtype=np.float32)
            
            # Convert to mono if stereo
            if len(audio.shape) > 1:
                audio = np.mean(audio, axis=1)
            
            return audio, sr
            
        except Exception as e:
            print(f"Error in load_audio_efficiently: {str(e)}")
            # Fallback: try to read in smaller chunks
            try:
                print("Trying chunked reading...")
                with sf.SoundFile(audio_path) as f:
                    # Read in 1-second chunks
                    chunk_size = f.samplerate  # 1 second
                    max_chunks = min(30, int(f.frames / chunk_size))
                    
                    audio_chunks = []
                    for i in range(max_chunks):
                        chunk = f.read(chunk_size, dtype=np.float32)
                        if len(chunk) == 0:
                            break
                        if len(chunk.shape) > 1:
                            chunk = np.mean(chunk, axis=1)
                        audio_chunks.append(chunk)
                    
                    if audio_chunks:
                        audio = np.concatenate(audio_chunks)
                        return audio, f.samplerate
                    else:
                        raise Exception("No audio data could be read")
                        
            except Exception as e2:
                print(f"Chunked reading also failed: {str(e2)}")
                raise e2
    
    def transcribe_single_file(self, audio_path: str) -> str:
        """Transcribe a single audio file with memory management."""
        try:
            # Force garbage collection before processing
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            # Load audio efficiently
            audio, sr = self.load_audio_efficiently(audio_path)
            print(f"Loaded audio: {len(audio)} samples at {sr} Hz")
            
            # Resample if needed
            if sr != 16000:
                try:
                    import librosa
                    audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
                    print("Resampled to 16kHz")
                except ImportError:
                    print("Warning: librosa not available. Assuming audio is 16kHz")
            
            # Ensure audio is the right length
            max_length = 30 * 16000  # 30 seconds at 16kHz
            if len(audio) > max_length:
                audio = audio[:max_length]
                print(f"Truncated to 30 seconds")
            
            # Process with the processor
            inputs = self.processor(
                audio, 
                sampling_rate=16000, 
                return_tensors="pt"
            )
            
            # Move to same device as model
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}
            
            # Generate transcription with proper Whisper configuration
            with torch.no_grad():
                # Create generation config without forced_decoder_ids conflict
                generation_config = self.model.generation_config
                generation_config.forced_decoder_ids = None
                generation_config.suppress_tokens = []
                
                generated_ids = self.model.generate(
                    inputs.input_features,
                    generation_config=generation_config,
                    max_length=448,
                    num_beams=1,  # Use greedy decoding for speed
                    do_sample=False,
                    use_cache=True
                )
            
            # Decode
            transcription = self.processor.batch_decode(
                generated_ids, 
                skip_special_tokens=True
            )[0]
            
            # Clean up memory immediately
            del inputs, generated_ids, audio
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            return transcription.strip()
            
        except Exception as e:
            print(f"Error processing {audio_path}: {str(e)}")
            # Clean up on error
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return ""
    
    def evaluate_dataset(self, data_folder: str) -> dict:
        """Evaluate the model on a dataset with memory-efficient processing."""
        results = {
            'files_processed': 0,
            'files_failed': 0,
            'references': [],
            'predictions': [],
            'file_results': []
        }
        
        # Get all wav files
        pattern = re.compile(r'medical_terminology_\d{1}_\d+\.wav$')
        wav_files = [f for f in os.listdir(data_folder) if pattern.match(f)]
        total_files = len(wav_files)
        
        print(f"Found {total_files} audio files to process")
        
        for i, wav_file in enumerate(wav_files):
            print(f"\nProcessing {i+1}/{total_files}: {wav_file}")
            
            # Force garbage collection before each file
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            audio_path = os.path.join(data_folder, wav_file)
            txt_file = wav_file.replace('.wav', '.txt')
            txt_path = os.path.join(data_folder, txt_file)
            
            # Check if corresponding text file exists
            if not os.path.exists(txt_path):
                print(f"Warning: No text file found for {wav_file}")
                results['files_failed'] += 1
                continue
            
            try:
                # Read reference text
                with open(txt_path, 'r', encoding='utf-8') as f:
                    reference = f.read().strip()
                
                # Generate prediction
                prediction = self.transcribe_single_file(audio_path)
                
                if prediction:  # Only process if transcription was successful
                    # Clean both texts
                    clean_ref = self.clean_text(reference)
                    clean_pred = self.clean_text(prediction)
                    
                    # Store results
                    results['references'].append(clean_ref)
                    results['predictions'].append(clean_pred)
                    results['file_results'].append({
                        'file': wav_file,
                        'reference': reference,
                        'prediction': prediction,
                        'clean_reference': clean_ref,
                        'clean_prediction': clean_pred
                    })
                    
                    results['files_processed'] += 1
                    
                    # Print progress
                    print(f"✓ Reference: {reference[:50]}...")
                    print(f"✓ Prediction: {prediction[:50]}...")
                else:
                    results['files_failed'] += 1
                    print(f"✗ Failed to transcribe {wav_file}")
                
            except Exception as e:
                print(f"Error processing {wav_file}: {str(e)}")
                results['files_failed'] += 1
            
            # Force cleanup after each file
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Calculate WER if we have results
        if results['references'] and results['predictions']:
            try:
                wer_score = wer(results['references'], results['predictions'])
                results['wer'] = wer_score
                print(f"\n🎯 Final Results:")
                print(f"Files processed: {results['files_processed']}")
                print(f"Files failed: {results['files_failed']}")
                print(f"Word Error Rate (WER): {wer_score:.4f}")
                print(f"Accuracy: {(1 - wer_score) * 100:.2f}%")
            except Exception as e:
                print(f"Error calculating WER: {str(e)}")
                results['wer'] = None
        else:
            print("No valid transcriptions to evaluate")
            results['wer'] = None
        
        return results
    
    def save_detailed_results(self, results: dict, output_file: str = "evaluation_results.txt"):
        """Save detailed evaluation results to a file."""
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("=== Whisper Model Evaluation Results ===\n\n")
            f.write(f"Files processed: {results['files_processed']}\n")
            f.write(f"Files failed: {results['files_failed']}\n")
            
            if results['wer'] is not None:
                f.write(f"Word Error Rate (WER): {results['wer']:.4f}\n")
                f.write(f"Accuracy: {(1 - results['wer']) * 100:.2f}%\n")
            
            f.write("\n=== Detailed Results ===\n\n")
            
            for result in results['file_results']:
                f.write(f"File: {result['file']}\n")
                f.write(f"Reference: {result['reference']}\n")
                f.write(f"Prediction: {result['prediction']}\n")
                f.write(f"Clean Ref: {result['clean_reference']}\n")
                f.write(f"Clean Pred: {result['clean_prediction']}\n")
                f.write("-" * 80 + "\n")
        
        print(f"Detailed results saved to {output_file}")

In [3]:
def run_memory_efficient_evaluation(
    model_path: str = "./whisper-finetuned",
    data_folder: str = "data_folder"
):
    """Run memory-efficient evaluation."""
    try:
        # Force initial cleanup
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        evaluator = MemoryEfficientEvaluator(model_path)
        results = evaluator.evaluate_dataset(data_folder)
        evaluator.save_detailed_results(results)
        return results
    except Exception as e:
        print(f"Evaluation failed: {str(e)}")
        return None


In [4]:
# Run evaluation
import torch, re, os, gc
from jiwer import wer
import soundfile as sf
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration

results = run_memory_efficient_evaluation(
    model_path="./whisper-finetuned",
    data_folder="data_folder"
)

if results and results['wer'] is not None:
    print(f"\n🎉 Evaluation completed successfully!")
    print(f"WER: {results['wer']:.4f}")
else:
    print("❌ Evaluation failed or no results obtained")

Loading model for evaluation...
Using CPU for inference
Found 250 audio files to process

Processing 1/250: medical_terminology_0_0.wav
Audio info: 1236992 frames, 44100 Hz, 28.05s
Loaded audio: 1236992 samples at 44100 Hz


`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [], 'begin_suppress_tokens': [220, 50256]}. If this is not desired, please set these values explicitly.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Resampled to 16kHz


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


✓ Reference: hHey everyone! JJ, here. In this lesson, I'm 
goin...
✓ Prediction: Hey everyone, JJ here. In this lesson I'm going to...

Processing 2/250: medical_terminology_0_1.wav
Audio info: 1368064 frames, 44100 Hz, 31.02s
Loaded audio: 1323000 samples at 44100 Hz
Resampled to 16kHz
✓ Reference: an end. The beginning is known as the prefix and 
...
✓ Prediction: And in the beginning is known as the prefix and th...

Processing 3/250: medical_terminology_0_10.wav
Audio info: 1941504 frames, 44100 Hz, 44.03s
Loaded audio: 1323000 samples at 44100 Hz
Resampled to 16kHz
✓ Reference: your arms away from your body or abduction, 
or ab...
✓ Prediction: moving your arms away from your body or abduction ...

Processing 4/250: medical_terminology_0_11.wav
Audio info: 2027520 frames, 44100 Hz, 45.98s
Loaded audio: 1323000 samples at 44100 Hz
Resampled to 16kHz
✓ Reference: for color is actually "chromo" or "chromato". "Leu...
✓ Prediction: for color is actually chroma or chromato. Luco, th...

In [ ]:
import os
import glob
from faster_whisper import WhisperModel
from jiwer import wer

# Configure model
model_size = "tiny.en"
model = WhisperModel(model_size)  # adjust based on your GPU

# Folder with audio and transcript files
DATA_FOLDER = "data_folder"

# Pattern for matching files
AUDIO_PATTERN = os.path.join(DATA_FOLDER, "medical_terminology_*_*.wav")

# Collect audio files
audio_files = sorted(glob.glob(AUDIO_PATTERN))
evaluator = MemoryEfficientEvaluator("./whisper-finetuned")


# Transcribe and evaluate
def evaluate_multiple_files(audio_files):
    all_refs = []
    all_hyps = []

    for audio_path in audio_files:
        base_name = os.path.splitext(os.path.basename(audio_path))[0]
        transcript_path = os.path.join(DATA_FOLDER, base_name + ".txt")

        # Check if corresponding transcript exists
        if not os.path.exists(transcript_path):
            print(f"[WARNING] Transcript missing for {audio_path}, Skipping...")
            continue

        # Read transcript
        with open(transcript_path, "r", encoding="utf-8") as f:
            ref_text = f.read().strip()

        # Transcribe audio
        segments, _ = model.transcribe(audio_path)
        hyp_text = " ".join(seg.text for seg in segments).strip()

        # Output
        print(f"\n--- {base_name} ---")
        print(f"Ref: {evaluator.clean_text(ref_text)}")
        print(f"Hyp: {evaluator.clean_text(hyp_text)}")

        all_refs.append(evaluator.clean_text(ref_text))
        all_hyps.append(evaluator.clean_text(hyp_text))

    # Compute WER
    if all_refs:
        score = wer(all_refs, all_hyps)
        print(f"\n=== Overall Word Error Rate (WER): {score:.2%} ===")
    else:
        print("\n[ERROR] No valid reference-transcription pairs found.")

# Run it
if __name__ == "__main__":
    evaluate_multiple_files(audio_files)



Loading model for evaluation...
Using CPU for inference

--- medical_terminology_0_0 ---
Ref: hhey everyone jj here in this lesson im going to be talking to you guys about medical terminology medical terminology is learning a different language and it pretty much is a different language but there are many easy ways to break the word apart in two different pieces to make it much easier to comprehend the first thing i want to talk to you guys about is just breaking it down in a medical term each word has a beginning a middle and
Hyp: hey everyone jj here in this lesson im going to be talking to you guys about medical terminology medical terminology is learning a different language its pretty much is a different language but there are many easy ways to break the word apart into different pieces to make it much easier to comprehend the first thing i want to talk to you guys about is just breaking it down in a medical term each word has a beginning a middle and then

--- medical_terminology

References

medical terminology [https://www.youtube.com/watch/04Wh2E9oNug]